# Ko-miracl RAG — 생성(Generation) 파이프라인

앞 노트북(`ko_miracl_rag.ipynb`)의 **임베딩 · ChromaDB · retriever** 코드를 재사용하고,
그 위에 **생성형 RAG** 파이프라인을 얹습니다. (LangChain 미사용)

```
사용자의 추상적 질문
   └─▶ ① LLM 질의 재작성 (rewrite)
        └─▶ ② retriever 검색 (ChromaDB, bge-m3)
             └─▶ ③ 프롬프트 생성 (검색 문서 주입)
                  └─▶ ④ LLM 응답 생성
```

- **임베딩(검색)**: `BAAI/bge-m3`
- **LLM(생성)**: `Qwen/Qwen2.5-7B-Instruct` (한국어 지원, Colab T4에서 4-bit 로드)

## 0. 설치 · 설정 (재사용 파트)

In [1]:
!pip -q install -U datasets chromadb sentence-transformers transformers accelerate bitsandbytes

In [2]:
import random, numpy as np, pandas as pd, torch
random.seed(42); np.random.seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# ===== 검색 임베딩 모델 (단일: bge-m3) =====
MODEL_CFG = {"model_name": "BAAI/bge-m3", "query_prefix": "", "passage_prefix": ""}

# ===== 생성 LLM (한국어 지원) =====
LLM_NAME = "Qwen/Qwen2.5-7B-Instruct"   # 메모리 부족 시 'Qwen/Qwen2.5-3B-Instruct' 로 교체

# ===== 하이퍼파라미터 =====
TOP_K = 5
N_EVAL_QUERIES = 30      # 인덱싱할 코퍼스 규모를 줄이려고 작게 설정
NEG_POOL_SIZE  = 3000
BATCH_SIZE = 128
MAX_SEQ_LEN = 512

Device: cuda


In [3]:
from datasets import load_dataset

# queries: 질문 목록 / default(dev): 관련도 라벨 → 데모용 인덱스 구성에 사용
q_dd = load_dataset("taeminlee/Ko-miracl", "queries")
queries = {r["_id"]: r["text"] for r in q_dd[list(q_dd.keys())[0]]}

default_dd = load_dataset("taeminlee/Ko-miracl", "default")
qrels_df = pd.DataFrame(default_dd["dev"])
print("queries:", len(queries), "| qrels(dev):", len(qrels_df))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


queries: 2761 | qrels(dev): 3057


### 코퍼스 부분집합 인덱싱 (재사용)
전체 149만 청크 대신 **정답 문서 + 네거티브 풀**만 스트리밍으로 수집합니다.
스트리밍이라 이 셀이 가장 오래 걸립니다 (수 분). 빠르게 보려면 `N_EVAL_QUERIES`를 줄이세요.

In [4]:
qrels_pos = qrels_df[qrels_df["score"] > 0]
eval_qids = qrels_pos["query-id"].unique().tolist()
random.shuffle(eval_qids); eval_qids = eval_qids[:N_EVAL_QUERIES]

gold = {}
for _, row in qrels_df[qrels_df["query-id"].isin(eval_qids)].iterrows():
    gold.setdefault(row["query-id"], {})[row["corpus-id"]] = float(row["score"])
needed_cids = {c for rels in gold.values() for c, s in rels.items() if s > 0}
print("데모 쿼리:", len(eval_qids), "| 정답 문서:", len(needed_cids))

데모 쿼리: 30 | 정답 문서: 100


In [5]:
corpus_dd = load_dataset("taeminlee/Ko-miracl", "corpus", streaming=True)
corpus_stream = corpus_dd[list(corpus_dd.keys())[0]]

docs, found, neg = {}, set(), 0
for row in corpus_stream:
    cid = row["_id"]
    if cid in needed_cids and cid not in docs:
        docs[cid] = {"title": row["title"], "text": row["text"]}; found.add(cid)
    elif neg < NEG_POOL_SIZE:
        docs[cid] = {"title": row["title"], "text": row["text"]}; neg += 1
    if len(found) >= len(needed_cids) and neg >= NEG_POOL_SIZE:
        break
print("수집 청크:", len(docs), "| 정답 확보:", len(found), "/", len(needed_cids))

수집 청크: 3100 | 정답 확보: 100 / 100


### 임베딩 · ChromaDB · retriever (재사용, 단일 모델)

In [6]:
from sentence_transformers import SentenceTransformer
import chromadb

def load_model(cfg):
    m = SentenceTransformer(cfg["model_name"], device=DEVICE); m.max_seq_length = MAX_SEQ_LEN; return m

def embed(model, texts, prefix, batch_size=BATCH_SIZE, show_progress=True):
    texts = [prefix + t for t in texts]
    return model.encode(texts, batch_size=batch_size, normalize_embeddings=True,
                        convert_to_numpy=True, show_progress_bar=show_progress)

chroma_client = chromadb.Client()

def build_collection(name, model, cfg, docs):
    try: chroma_client.delete_collection(name)
    except Exception: pass
    col = chroma_client.create_collection(name=name, metadata={"hnsw:space": "cosine"})
    cids = list(docs.keys())
    texts = [docs[c]["text"] for c in cids]; titles = [docs[c]["title"] for c in cids]
    embs = embed(model, texts, cfg["passage_prefix"])
    for i in range(0, len(cids), BATCH_SIZE):
        col.add(ids=cids[i:i+BATCH_SIZE], embeddings=embs[i:i+BATCH_SIZE].tolist(),
                documents=texts[i:i+BATCH_SIZE],
                metadatas=[{"title": t} for t in titles[i:i+BATCH_SIZE]])
    print(f"[{name}] 적재 완료: {col.count()} chunks"); return col

def retrieve(col, model, cfg, query_text, k=TOP_K):
    q_emb = embed(model, [query_text], cfg["query_prefix"], batch_size=1, show_progress=False)
    res = col.query(query_embeddings=q_emb.tolist(), n_results=k)
    out = []
    for cid, doc, dist, meta in zip(res["ids"][0], res["documents"][0],
                                    res["distances"][0], res["metadatas"][0]):
        out.append({"corpus_id": cid, "score": 1 - dist,          # score = 1 - distance
                    "title": meta.get("title", ""), "text": doc})  # 생성용이라 전체 텍스트 유지
    return out

In [15]:
model = load_model(MODEL_CFG)
collection = build_collection("ko-miracl-bge-m3", model, MODEL_CFG, docs)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

[ko-miracl-bge-m3] 적재 완료: 3100 chunks


## 1. 생성 LLM 로드 (한국어)
`Qwen2.5-7B-Instruct`를 4-bit로 로드합니다.

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, quantization_config=bnb, device_map="auto")
llm.eval()
print("LLM loaded:", LLM_NAME)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

LLM loaded: Qwen/Qwen2.5-7B-Instruct


In [8]:
@torch.no_grad()
def chat(system, user, max_new_tokens=512, temperature=0.3):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    # 최신 transformers는 return_dict=True로 BatchEncoding을 받아 **enc로 넘기는 게 안전
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(llm.device)
    out = llm.generate(**enc, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=max(temperature, 1e-5),
                       top_p=0.9, pad_token_id=tok.eos_token_id)
    gen = out[0][enc["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

## 2. ① 질의 재작성 (Query Rewriting)
사용자의 추상적/모호한 질문을 검색에 적합한 핵심 질의로 재작성합니다.

In [9]:
def rewrite_query(question):
    system = ("당신은 검색 질의 재작성 전문가입니다. 사용자의 모호하거나 추상적인 질문을 "
              "검색엔진에 적합하도록 핵심 개체·키워드 중심의 명확한 한국어 검색 질의로 바꾸세요. "
              "설명 없이 재작성된 질의 한 줄만 출력하세요.")
    user = f"사용자 질문: {question}\n재작성된 검색 질의:"
    return chat(system, user, max_new_tokens=64, temperature=0.0)

## 3. ③ 프롬프트 생성
검색된 문서를 근거로 주입하는 RAG 프롬프트를 만듭니다.

In [ ]:
SYSTEM_GEN = ("당신은 주어진 참고 문서를 근거로 한국어로 답하는 assistant입니다. "
              "문서에 없는 내용은 지어내지 말고, 근거가 없으면 '제공된 문서에서 찾을 수 없습니다'라고 답하세요. "
              "답변에 사용한 문서 번호를 [문서 n] 형태로 인용하세요.")

def _format_ctx(contexts, snippet_len=800):
    return "\n\n".join(
        f"[문서 {i+1}] (id={c['corpus_id']}, title={c['title']})\n{c['text'][:snippet_len]}"
        for i, c in enumerate(contexts))

# ── A) 베이스라인: 지시만 (기존 프롬프트) ──
def build_prompt(question, contexts, snippet_len=800):
    user = f"# 참고 문서\n{_format_ctx(contexts, snippet_len)}\n\n# 질문\n{question}\n\n# 답변"
    return SYSTEM_GEN, user

# ── B) Few-shot: 이상적 답변 1개 + 근거없음→거절 1개를 예시로 주입 ──
_FEWSHOT = """다음은 답변 형식 예시입니다.

# 예시 1
# 참고 문서
[문서 1] (id=ex-a, title=에베레스트산)
에베레스트산은 해발 8,848m로 지구에서 가장 높은 산이다.
# 질문
세계에서 가장 높은 산의 높이는?
# 답변
세계에서 가장 높은 산인 에베레스트산의 높이는 해발 8,848m입니다. [문서 1]

# 예시 2 (문서에 근거 없음 → 지어내지 말고 거절)
# 참고 문서
[문서 1] (id=ex-b, title=커피)
커피는 커피나무 열매의 씨앗을 볶아 만든 음료다.
# 질문
녹차에 들어있는 카페인 함량은?
# 답변
제공된 문서에서 찾을 수 없습니다.

이제 아래 실제 질문에 위 형식으로 답하세요.
"""

def build_prompt_fewshot(question, contexts, snippet_len=800):
    user = (_FEWSHOT
            + f"\n# 참고 문서\n{_format_ctx(contexts, snippet_len)}"
            + f"\n\n# 질문\n{question}\n\n# 답변")
    return SYSTEM_GEN, user

## 4. 전체 파이프라인
사용자 질문 → ① 재작성 → ② 검색 → ③ 프롬프트 → ④ 응답

In [ ]:
def rag_answer(question, k=TOP_K, verbose=True, prompt_fn=build_prompt):
    # ① 질의 재작성
    rq = rewrite_query(question)
    # ② retriever 검색
    ctxs = retrieve(collection, model, MODEL_CFG, rq, k=k)
    # ③ 프롬프트 생성 (prompt_fn 으로 A/B 교체)
    system, user = prompt_fn(question, ctxs)
    # ④ LLM 응답 생성
    answer = chat(system, user, max_new_tokens=512, temperature=0.2)

    if verbose:
        print("① 원 질문     :", question)
        print("① 재작성 질의 :", rq)
        print("② 검색 결과 (score = 1 - distance):")
        for i, c in enumerate(ctxs, 1):
            print(f"   [{i}] score={c['score']:.4f} id={c['corpus_id']} | {c['title']}")
        print("\n④ 생성 응답 :\n" + answer)
    return {"question": question, "rewritten": rq, "contexts": ctxs, "answer": answer}

## 5. 실행 예시

In [16]:
# 예시 A: Ko-miracl dev의 실제 질문 (정답 문서가 인덱스에 포함되어 있음)
_ = rag_answer(queries[eval_qids[0]])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


① 원 질문     : 구지가는 고대 시가의 하나인가요?
① 재작성 질의 : 구지가가 고대 시인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.6918 id=264838#12 | 한국의 상고시대 시가
   [2] score=0.4292 id=986864#0 | 양해 (후한)
   [3] score=0.4098 id=987124#0 | 요시카와 코지로
   [4] score=0.3878 id=989604#0 | 트라시마코스
   [5] score=0.3871 id=988540#71 | 왕자 조

④ 생성 응답 :
네, 구지가는 고대 시가의 하나입니다. [문서 1]에 따르면, 〈구지가(龜旨歌)〉는 서기 40년경에 이루어진 고대 시가의 하나라고 기록되어 있습니다.


In [17]:
_ = rag_answer(queries[eval_qids[12]])

① 원 질문     : 한국어는 언제 만들어졌나요?
① 재작성 질의 : 한국어가 언제 시작되었나요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.6093 id=487516#0 | 한글
   [2] score=0.4787 id=1852#9 | 유럽
   [3] score=0.4708 id=261019#0 | 불교의 역사
   [4] score=0.4509 id=987704#3 | 학생신문
   [5] score=0.4459 id=3966#1 | 한국의 성씨와 이름

④ 생성 응답 :
한국어를 표기하기 위해 만들어진 문자는 1446년에 창제되었습니다. [문서 1]에 따르면, 세종대왕이 훈민정음(훈민정음이란 이름으로 알려짐)을 창제하고 1446년에 이를 반포하였습니다.


In [18]:
# 예시 B: 사용자가 던지는 추상적/모호한 질문
_ = rag_answer("그 유명한 상대성 이론 만든 사람 어떤 인물이야?")

① 원 질문     : 그 유명한 상대성 이론 만든 사람 어떤 인물이야?
① 재작성 질의 : 상대성 이론을 제안한 사람 누구인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.4527 id=3533#0 | 르네 데카르트
   [2] score=0.4490 id=985079#3 | 마이어-피토리스 열
   [3] score=0.4317 id=780040#17 | 라이트 판타스틱
   [4] score=0.4213 id=987149#3 | 제임스 워델 알렉산더
   [5] score=0.4148 id=987221#0 | 류드비크 파데예프

④ 생성 응답 :
제공된 문서에서 찾을 수 없습니다. 상대성 이론은 아인슈타인에 의해 만들어졌지만, 해당 참고 문서에는 관련 정보가 없습니다.


In [19]:
_ = rag_answer("물 몇도에 끓음?")

① 원 질문     : 물 몇도에 끓음?
① 재작성 질의 : 물이 끓는 온도는 몇 도인가요?
② 검색 결과 (score = 1 - distance):
   [1] score=0.5234 id=986668#2 | 2003년 유럽 폭염
   [2] score=0.5144 id=985918#0 | 로스팅
   [3] score=0.4879 id=987625#13 | 주위
   [4] score=0.4788 id=47536#1 | 우라늄
   [5] score=0.4788 id=986668#1 | 2003년 유럽 폭염

④ 생성 응답 :
제공된 문서에서 찾을 수 없습니다.


In [20]:
_ = rag_answer("키보드 주문제작 하고 싶어")

① 원 질문     : 키보드 주문제작 하고 싶어
① 재작성 질의 : 키보드 제작 서비스 찾기
② 검색 결과 (score = 1 - distance):
   [1] score=0.6641 id=985068#3 | 커스텀 키보드
   [2] score=0.6520 id=985068#4 | 커스텀 키보드
   [3] score=0.6363 id=985068#2 | 커스텀 키보드
   [4] score=0.6326 id=985068#0 | 커스텀 키보드
   [5] score=0.6277 id=985068#1 | 커스텀 키보드

④ 생성 응답 :
키보드 주문제작을 원하신다면 두 가지 방법이 있습니다. 하나는 직접 설계와 제작하는 방법이고, 다른 하나는 공동으로 제작하는 방법입니다. 

직접 설계 및 제작하는 방법은 키보드 케이스, 기판, 보강판 등을 직접 설계하고 제작해야 하며, 이 과정에서는 전문 지식이 필요합니다. 또한, 이 방법은 비용이 많이 들기 때문에 일부 사용자들은 이를 선택하지 못할 수도 있습니다. [문서 1] 및 [문서 2]에 따르면, 이 방법은 전문 지식이 필요하고 비용이 많이 들기 때문에 몇몇 사용자들을 제외하고는 현실적으로 실현하기 어렵습니다.

반면, 공동으로 제작하는 방법은 전문적인 지식 없이 비교적 저렴하게 커스텀 키보드를 제작할 수 있습니다. 이 방법은 여러 사용자들이 모여 각각의 역할을 분담하여 작업을 진행하며, 이로 인해 비용이 절감되고 전문 지식이 필요한 부분도 줄어듭니다. [문서 2]에 따르면, 공동 제작을 통해 전문적인 지식 없이 비교적 저렴하게 커스텀 키보드를 제작할 수 있습니다.

따라서, 키보드 주문제작을 원하신다면 공동으로 제작하는 방법을 고려해보시는 것이 좋을 것 같습니다.


## 6. 프롬프트 A/B 비교 (지표 기반 채택)
같은 검색 결과 위에서 **A(지시형) vs B(few-shot)** 응답을 생성해 비교합니다.
검색은 프롬프트와 무관하므로 쿼리당 한 번만 수행해 공유하고, 생성 프롬프트만 교체합니다.

| 지표 | 의미 | 방향 |
|---|---|---|
| **인용recall(정답검색시)** | 정답 문서가 검색된 쿼리에서, 답변이 그 정답 문서를 `[문서 n]`으로 인용한 비율 | 높을수록 좋음 |
| **거절율(정답없을때)** | 정답 문서가 검색되지 않은 쿼리에서, 지어내지 않고 거절한 비율 | 높을수록 좋음 |
| **포맷준수율** | 인용 또는 거절 형식을 지킨 비율 | 높을수록 좋음 |

세 지표에서 앞서는 프롬프트를 `rag_answer(..., prompt_fn=...)` 기본값으로 채택하면 됩니다.

In [ ]:
import re
from tqdm.auto import tqdm

def _cited_ids(answer, ctxs):
    # 답변의 [문서 n] → 실제 corpus_id 집합
    idxs = {int(n) for n in re.findall(r"\[문서\s*(\d+)\]", answer)}
    return {ctxs[i-1]["corpus_id"] for i in idxs if 1 <= i <= len(ctxs)}

def eval_prompts(prompt_fns, qids, k=TOP_K, max_new_tokens=256):
    """같은 검색 결과 위에서 프롬프트만 바꿔 생성 → 지표 비교.
    (검색은 프롬프트와 무관하므로 쿼리당 1번만 수행해 공유)"""
    rows = {name: [] for name in prompt_fns}
    bar = tqdm(qids, desc="A/B 평가")
    for qid in bar:
        q = queries[qid]
        ctxs = retrieve(collection, model, MODEL_CFG, rewrite_query(q), k=k)
        retrieved = [c["corpus_id"] for c in ctxs]
        gold_cids = {cid for cid, s in gold.get(qid, {}).items() if s > 0}
        gold_hit = gold_cids & set(retrieved)   # 검색이 정답을 물어왔나 (A/B 공유)
        for name, fn in prompt_fns.items():
            bar.set_postfix_str(f"{name} 생성중")
            system, user = fn(q, ctxs)
            ans = chat(system, user, max_new_tokens=max_new_tokens, temperature=0.2)
            cited = _cited_ids(ans, ctxs)
            abstained = "찾을 수 없습니다" in ans
            rows[name].append({
                "answerable": bool(gold_hit),
                "cite_gold": bool(cited & gold_hit),      # 정답 문서를 인용했나
                "abstained": abstained,
                "fmt_ok": bool(cited) or abstained,        # 인용 or 거절 형식 준수
            })
    return rows

def summarize(rows):
    out = []
    for name, rs in rows.items():
        ans_q = [r for r in rs if r["answerable"]]      # 정답이 검색된 쿼리
        no_q  = [r for r in rs if not r["answerable"]]  # 정답이 안 검색된 쿼리
        out.append({
            "prompt": name,
            "인용recall(정답검색시)": round(np.mean([r["cite_gold"] for r in ans_q]), 3) if ans_q else None,
            "거절율(정답없을때)":     round(np.mean([r["abstained"] for r in no_q]), 3) if no_q else None,
            "포맷준수율":             round(np.mean([r["fmt_ok"] for r in rs]), 3),
            "n": len(rs),
        })
    return pd.DataFrame(out)

# 먼저 5개로 파이프라인·ETA 확인 → 괜찮으면 아래 줄을 eval_qids 로 바꿔 전체 실행
rows = eval_prompts({"A_baseline": build_prompt, "B_fewshot": build_prompt_fewshot}, eval_qids[:5])
summarize(rows)

### 더 해볼 것
- 재작성을 **다중 질의(multi-query)** 로 확장해 검색 recall 향상
- 리랭커(`BAAI/bge-reranker-v2-m3`)로 검색 결과 재정렬 후 상위 문서만 프롬프트에 주입
- 답변의 인용 문서와 정답셋(`gold`)을 비교해 groundedness 평가
- Qwen vs GPT 비교